In [ ]:
import time
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.utils import gen_batches
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from nn.tensor import Tensor
from nn.optim.SGD import SGD
from nn.utils import visualize_computation_graph
from MnistModel import MNISTModel, MNISTConvModel
from nn.loss.crossentropyloss import cross_entropy

c:\Users\NSU Student\anaconda3\envs\ml\Lib\site-packages\cupy\_environment.py:286: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [2]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)

x = mnist.data
y = mnist.target

In [3]:
x_train_all, y_train_all = x[:60000], y[:60000]
x_test, y_test = x[60000:], y[60000:]

In [4]:
x_train_all = x_train_all.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0
y_train_all = y_train_all.astype(np.int64)
y_test = y_test.astype(np.int64)

In [5]:
x_train, x_validation, y_train, y_validation = train_test_split(x_train_all, y_train_all, test_size=0.1, random_state=42, shuffle=True)

In [6]:
print(f"Training examples: {len(x_train)}")
print(f"Validation examples: {len(y_validation)}")
print(f"Test examples: {len(x_test)}")

Training examples: 54000
Validation examples: 6000
Test examples: 10000


In [15]:
model = MNISTConvModel()
optimizer = SGD(model.parameters(), lr=0.01)

In [16]:
epochs = 10
batch_size = 64

In [ ]:
training_loss, validation_loss = [], []

for epoch in range(epochs): 
    epoch_loss = 0
    training_examples = 0
    total_train_batches = (len(x_train) + batch_size - 1) // batch_size
    pbar = tqdm(gen_batches(len(x_train), batch_size=batch_size), total=total_train_batches, desc=f"Epoch {epoch+1:2d}/{epochs:2d}", leave=True)
    curr_time = time.time()
    for batch_idx in pbar: 
        batch_x = Tensor(x_train[batch_idx], requires_grad=True)
        batch_y = Tensor(y_train[batch_idx], requires_grad = True)

        logits = model(batch_x)
        loss = cross_entropy(logits, batch_y)
        loss.zero_grad()
        loss.backward()
        optimizer.step()

        count = batch_idx.stop - batch_idx.start
        epoch_loss += float(loss.data) * count
        training_examples += count
    training_time = time.time() - curr_time
    training_loss.append(epoch_loss / training_examples)
    pbar.set_postfix({"Training Loss": f"{training_loss[-1]:.4f}"})

    epoch_val_loss = 0.0
    val_examples = 0

    curr_time = time.time()
    for val_batch_idx in gen_batches(len(x_validation), batch_size = batch_size): 
        val_batch_x = Tensor(x_validation[val_batch_idx], requires_grad=False)
        val_batch_y = Tensor(y_validation[val_batch_idx], requires_grad=False)
        val_logits = model(val_batch_x)
        val_loss = cross_entropy(val_logits, val_batch_y)
        val_count = val_batch_idx.stop - val_batch_idx.start
        epoch_val_loss += float(val_loss.data) * val_count
        val_examples += val_count
    validation_loss.append(epoch_val_loss / val_examples)
    pbar.set_postfix({"Validation Loss": f"{validation_loss[-1]:.4f}"})
    validation_time = time.time() - curr_time

    print("Epoch %2d/%2d ==> Training Loss: %4f and Validation Loss: %4f : Trianing time: %2f and Validation time: %2f" % (epoch, epochs, training_loss[-1], validation_loss[-1], training_time/60, validation_time/60))
    

Epoch  1/10:   0%|          | 0/844 [00:00<?, ?it/s]

Epoch  1/10: 100%|██████████| 844/844 [01:37<00:00,  8.67it/s]


Epoch  0/10 ==> Training Loss: 1.856664 and Validation Loss: 0.510222 : Trianing time: 1.621791 and Validation time: 0.127809


Epoch  2/10: 100%|██████████| 844/844 [01:31<00:00,  9.18it/s]


Epoch  1/10 ==> Training Loss: 0.404598 and Validation Loss: 0.324432 : Trianing time: 1.531817 and Validation time: 0.123606


...

Epoch  5/10: 100%|██████████| 844/844 [01:49<00:00,  7.69it/s]


Epoch  4/10 ==> Training Loss: 0.172936 and Validation Loss: 0.152433 : Trianing time: 1.829298 and Validation time: 0.170890


Epoch  6/10: 100%|██████████| 844/844 [01:49<00:00,  7.69it/s]


Epoch  5/10 ==> Training Loss: 0.145790 and Validation Loss: 0.132064 : Trianing time: 1.829653 and Validation time: 0.140655


Epoch  7/10:  24%|██▍       | 206/844 [00:24<02:34,  4.12it/s]

In [ ]:
def visualize(): 
    v_x, v_y = Tensor(x_train[:batch_size], requires_grad = True), Tensor(y_train[:batch_size], requires_grad = True)
    logits = model(v_x)
    loss = cross_entropy(logits, v_y)
    return visualize_computation_graph(loss, "computation_graph.png")

In [ ]:
visualize()

In [ ]:
plt.figure(figsize=(8, 5))
epoch_numbers = range(1, epochs + 1)
plt.plot(epoch_numbers, training_loss, marker="o", label="Training loss")
plt.plot(epoch_numbers, validation_loss, marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Training and Validation Loss")
plt.xticks(list(epoch_numbers))
plt.grid(True, alpha=0.6)
plt.legend()
plt.show()

In [ ]:
test_loss_sum = 0.0
test_examples = 0

for test_batch_idx in gen_batches(len(x_test), batch_size=batch_size):
    x_batch = Tensor(x_test[test_batch_idx], requires_grad=False)
    y_batch = Tensor(y_test[test_batch_idx], requires_grad=False)
    test_logits = model(x_batch)

    count = test_batch_idx.stop - test_batch_idx.start
    test_loss_sum += float(cross_entropy(test_logits, y_batch).data) * count
    test_examples += count

print(f"Test loss: {test_loss_sum / test_examples:.4f}")

In [ ]:
def predict(x, y, start, end):
    start = max(0, min(start, len(y)))
    end = max(start, min(end, len(y)))
    if start == end:
        raise ValueError("The requested image range is empty.")

    images = x[start:end]
    actual = y[start:end].astype(int)
    predictions = np.argmax(model(Tensor(images)).data, axis=1)

    cols = 8
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2.2))
    axes = np.atleast_1d(axes).ravel()

    for index, (image, target, prediction) in enumerate(zip(images, actual, predictions)):
        ax = axes[index]
        ax.imshow(image.reshape(28, 28), cmap="gray")
        ax.axis("off")
        ax.text(
            0.03, 0.97, f"Prediction: {prediction}",
            transform=ax.transAxes, ha="left", va="top",
            color="black", fontsize=8,
            bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none", "pad": 1.5},
        )
        ax.text(
            0.97, 0.97, f"Actual: {target}",
            transform=ax.transAxes, ha="right", va="top",
            color="black", fontsize=8,
            bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none", "pad": 1.5},
        )
        if prediction != target:
            ax.add_patch(
                plt.Rectangle(
                    (0, 0), 1, 1, transform=ax.transAxes,
                    fill=False, edgecolor="red", linewidth=3, clip_on=False,
                )
            )

    for ax in axes[len(images):]:
        ax.axis("off")

    plt.show()


In [ ]:
predict(x_test, y_test, 0, 100)
